# Evaluación del Recuperador RAG (Retrieval-Augmented Generation)

Se presenta la evaluación cuantitativa de la calidad de recuperación del índice vectorial (FAISS) diseñado para el Agente Conversacional. Se somete al sistema a **10 consultas de prueba (queries)** diseñadas específicamente para cubrir la casuística de los cinco pilares temáticos del corpus de conocimiento.

Para medir el rendimiento, se calculan las siguientes métricas de evaluación de sistemas de recuperación de información (Information Retrieval):
* **Precision@3 (Semántica):** Fracción de los *top-3 chunks* recuperados por similitud semántica cuyo pilar temático coincide con el esperado para la consulta.
* **Hit@Any:** Variable booleana que indica si al menos un resultado (ya sea por similitud semántica o inyectado por la regla de seguridad *Failsafe*) pertenece al pilar esperado.
* **Global Precision@3:** Media aritmética de la métrica *Precision@3* calculada sobre el conjunto de las 10 consultas.

## 0. Imports y configuración

In [ ]:
import os
import sys
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

def save_custom_plot(filename, dpi=300):
    """
    Guarda la figura activa de matplotlib en el directorio de resultados.
    Asegura alta resolución y recorta los márgenes blancos innecesarios.
    """
    PROJECT_ROOT = Path('/home/adrian/projects/chatbots/chatbot-ciberacoso')
    output_dir = PROJECT_ROOT / 'docs' / 'figures'
    
    if not filename.endswith('.png'):
        filename += '.png'
        
    plt.savefig(output_dir / filename, bbox_inches='tight', dpi=dpi)
    print(f"Gráfica guardada: {filename}")

# Forzamos la ruta absoluta desde la raíz del proyecto
PROJECT_ROOT = Path("..").resolve() 
vector_dir = PROJECT_ROOT / "data" / "vectorstore" / "faiss_index_v1"

# Cambio de directorio activo para asegurar que las rutas relativas (ej. data/) funcionen
os.chdir(str(PROJECT_ROOT))

# Estilo visual consistente con los notebooks anteriores
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

## 1. Consultas de Prueba

In [ ]:
TEST_CASES = [
    {
        "label": "Q1",
        "query": "tengo miedo de ir al instituto porque me amenazan por internet",
        "emotion": "fear",
        "expected_pillars": {1, 2},
        "description": "Miedo + amenazas --> psicoeducación + TCC",
    },
    {
        "label": "Q2",
        "query": "me han publicado fotos mías sin permiso y no sé qué hacer",
        "emotion": "fear",
        "expected_pillars": {1, 5},
        "description": "Sexting/sextorsión --> protocolo + autodefensa digital",
    },
    {
        "label": "Q3",
        "query": "no puedo parar de llorar me siento muy mal y no sé por qué",
        "emotion": "sadness",
        "expected_pillars": {2},
        "description": "Tristeza intensa --> técnicas TCC",
    },
    {
        "label": "Q4",
        "query": "todo el mundo me odia en clase y en los grupos de whatsapp me ignoran",
        "emotion": "sadness",
        "expected_pillars": {2},
        "description": "Distorsión cognitiva todo/nada --> TCC",
    },
    {
        "label": "Q5",
        "query": "quiero denunciar al que me acosa pero no sé cómo hacerlo",
        "emotion": "anger",
        "expected_pillars": {1},
        "description": "Denuncia --> protocolo P1",
    },
    {
        "label": "Q6",
        "query": "cómo bloqueo a alguien en instagram que me está insultando",
        "emotion": "others",
        "expected_pillars": {5},
        "description": "Autodefensa digital Instagram --> P5",
    },
    {
        "label": "Q7",
        "query": "todo lo que hago sale mal y no sirvo para nada",
        "emotion": "sadness",
        "expected_pillars": {3},
        "description": "Baja autoestima --> hope-focused / contranarrativa",
    },
    {
        "label": "Q8",
        "query": "me van a pegar si no hago lo que dicen me da muchísimo miedo",
        "emotion": "fear",
        "expected_pillars": {4},
        "description": "Amenaza física urgente --> PAP crisis",
    },
    {
        "label": "Q9",
        "query": "ya no quiero seguir viviendo nadie me va a echar de menos",
        "emotion": "sadness",
        "expected_pillars": {4},
        "description": "Ideación suicida --> PAP crisis",
    },
    {
        "label": "Q10",
        "query": "me da mucha vergüenza lo que pasó y no puedo contárselo a nadie",
        "emotion": "disgust",
        "expected_pillars": {2},
        "description": "Vergüenza --> normalización TCC",
    },
]

## 2. Inicializar el Recuperador RAG

In [ ]:
from src.rag.document_ingestion_v1 import RAGRetriever

print("Inicializando RAG Retriever (FAISS + SentenceTransformers)...")
retriever = RAGRetriever(vectorstore_dir=vector_dir)

print("RAGRetriever listo.")
print(f"  Vectorstore: {retriever.vectorstore_dir}")
print(f"  Chunks totales: {retriever.index.ntotal}")
print(f"  always_include: {len(retriever._always_include_indices)} chunk(s)")

## 3. Resultados

Para cada consulta se muestran:
- Los **top-3 chunks semánticos** con su pilar, score y si coincide con el esperado (`V` / `·`).
- Los **chunks forzados** (always_include) añadidos por la regla de emoción.
- Tabla y visualización resumen: Verde ≥ 0.67 · Amarillo ≥ 0.34 · Rojo < 0.34

In [ ]:
PILLAR_NAMES = {
    1: "Protocolos",
    2: "TCC",
    3: "Esperanza",
    4: "PAP",
    5: "Autodefensa",
}

all_results = []

print("Trazabilidad de Recuperación por Consulta")

for tc in TEST_CASES:
    # Recuperación híbrida (semántica + forzada por emoción)
    results = retriever.retrieve(query=tc["query"], emotion=tc["emotion"], top_k=3)

    # Separación de resultados para cálculo de métricas
    semantic = [r for r in results if not r.forced]
    forced   = [r for r in results if r.forced]

    # Cálculo de métricas
    hits    = sum(1 for r in semantic if r.chunk.pillar in tc["expected_pillars"])
    p_at_3  = hits / 3  # Asumimos k=3 fijo según configuración
    hit_any = any(r.chunk.pillar in tc["expected_pillars"] for r in results)

    all_results.append(
        dict(tc=tc, results=results, semantic=semantic, forced=forced,
             p_at_3=p_at_3, hits=hits, hit_any=hit_any)
    )

    # Formateo de salida por consola
    exp_str = ", ".join(f"P{p}" for p in sorted(tc["expected_pillars"]))
    status  = "V" if hit_any else "X"
    print(f"{'-'*70}")
    print(f"{tc['label']}  [{tc['emotion']:<7}]  precision@3={p_at_3:.2f}  {status}")
    print(f"  Query    : {tc['query']}")
    print(f"  Esperado : {exp_str} - {tc['description']}")
    
    print(f"  Semántico:")
    for i, r in enumerate(semantic, 1):
        m = "V" if r.chunk.pillar in tc["expected_pillars"] else "·"
        pname = PILLAR_NAMES[r.chunk.pillar]
        print(f"    {i}. [{r.chunk.id}] P{r.chunk.pillar}-{pname:<11} "
              f"score={r.score:.4f}  {m}  {r.chunk.title[:48]}...")
              
    if forced:
        print(f"  Forzados (Failsafe):")
        for r in forced:
            pname = PILLAR_NAMES[r.chunk.pillar]
            print(f"    --> [{r.chunk.id}] P{r.chunk.pillar}-{pname}  (always_include)")

print(f"{'-'*70}\n")

In [ ]:
summary_rows = []
for res in all_results:
    tc = res["tc"]
    exp_str = ", ".join(f"P{p}" for p in sorted(tc["expected_pillars"]))
    sem_pillars = [r.chunk.pillar for r in res["semantic"]]
    forced_ids = [r.chunk.id for r in res["forced"]]
    summary_rows.append(
        {
            "Consulta": tc["label"],
            "Emoción": tc["emotion"].capitalize(),
            "Esperado": exp_str,
            "Top-3 Pilares": str(sem_pillars),
            "Forzados": ", ".join(forced_ids) if forced_ids else "-",
            "Precision@3": res["p_at_3"],
            "Hit@Any": "V" if res["hit_any"] else "X",
        }
    )

df_summary = pd.DataFrame(summary_rows)
global_p3 = df_summary["Precision@3"].mean()
hit_any_rate = (df_summary["Hit@Any"] == "V").mean()

print(f"Global Precision@3 = {global_p3:.4f}  ({global_p3*100:.1f}%)")
print(f"Global Hit@Any = {hit_any_rate:.4f}  ({hit_any_rate*100:.1f}%)\n")

# Estilos dinámicos para la tabla
def _color_p3(val: float) -> str:
    if val >= 0.67:
        return "background-color: #d4edda; color: #155724"  # Verde
    if val >= 0.34:
        return "background-color: #fff3cd; color: #856404"  # Amarillo
    return "background-color: #f8d7da; color: #721c24"      # Rojo

styled = (
    df_summary.style
    .pipe(lambda s: s.map(_color_p3, subset=["Precision@3"])
          if hasattr(s, "map") else s.applymap(_color_p3, subset=["Precision@3"]))
    .map(lambda v: 'color: #155724; font-weight: bold' if v == 'V' else 'color: #721c24; font-weight: bold', subset=['Hit@Any'])
    .format({"Precision@3": "{:.2f}"})
    .set_caption(
        f"<b>Resumen de Rendimiento RAG</b><br>"
        f"Global Precision@3 = {global_p3:.3f} | Hit@Any = {hit_any_rate:.2f}"
    )
    .hide(axis="index")
)
display(styled)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle(
    "RAG Retriever — Evaluación de Recuperación",
    fontsize=14,
    fontweight="bold",
    y=1.05,
)

# Panel izquierdo: Precision@3 por consulta
ax1 = axes[0]
labels = [r["tc"]["label"] for r in all_results]
p3_vals = [r["p_at_3"] for r in all_results]

# Colores semánticos: Verde (>=0.67), Naranja (>=0.34), Rojo (<0.34)
bar_col = ["#2ecc71" if v >= 0.67 else "#f39c12" if v >= 0.34 else "#e74c3c" for v in p3_vals]

ax1.bar(labels, p3_vals, color=bar_col, edgecolor="white", linewidth=0.8)
ax1.axhline(
    global_p3,
    color="navy",
    linewidth=1.5,
    linestyle="--",
    label=f"Media = {global_p3:.2f}",
)
ax1.set_ylim(0, 1.18)
ax1.set_xlabel("Consulta de Prueba", fontweight="bold")
ax1.set_ylabel("Precision@3", fontweight="bold")
ax1.set_title("Eficacia Semántica por Consulta", pad=15)
ax1.legend(fontsize=9)
ax1.tick_params(axis="x", labelsize=9)

# Anotaciones de valor sobre las barras
for x, v in enumerate(p3_vals):
    ax1.text(x, v + 0.02, f"{v:.2f}", ha="center", va="bottom", fontsize=9, fontweight="bold")
sns.despine(ax=ax1)

# Panel derecho: Distribución de pilares recuperados
ax2 = axes[1]
pillar_counts = Counter()

# Contabilizamos solo las recuperaciones puramente semánticas
for res in all_results:
    for r in res["semantic"]:
        key = f"P{r.chunk.pillar}\n{PILLAR_NAMES[r.chunk.pillar]}"
        pillar_counts[key] += 1

pilar_labels = sorted(pillar_counts.keys())
pilar_vals   = [pillar_counts[k] for k in pilar_labels]

# Paleta de colores para diferenciar los pilares
panel_colors = ["#3498db", "#9b59b6", "#1abc9c", "#e67e22", "#e74c3c"]

ax2.bar(
    pilar_labels,
    pilar_vals,
    color=panel_colors[: len(pilar_labels)],
    edgecolor="white",
    linewidth=0.8,
)
ax2.set_xlabel("Pilar Temático", fontweight="bold")
ax2.set_ylabel("Nº apariciones en Top-3 Semántico", fontweight="bold")
ax2.set_title("Frecuencia de Recuperación por Pilar", pad=15)

for x, v in enumerate(pilar_vals):
    ax2.text(x, v + 0.1, str(v), ha="center", va="bottom", fontsize=10, fontweight="bold")
sns.despine(ax=ax2)

plt.tight_layout()

# Usamos la función centralizada para mantener todo en docs/figures
save_custom_plot('rag_test_precision_distribution')
plt.show()

## 4. Análisis de resultados

**Global Precision@3 = 0.50 | Hit@Any = 0.90**

### Consultas con Precision@3 = 1.0

- **Q6** (bloqueo Instagram): Recuperación perfecta. El Pilar 5 domina completamente el espacio semántico de las acciones técnicas en redes sociales, sin solaparse con los pilares psicológicos.

### Consultas con Precision@3 ≥ 0.67

- **Q1** (miedo + amenazas): 2/3 correctos (P1). Los resultados acertados se enfocan en los protocolos de actuación. El resultado "erróneo" recupera un chunk de amenaza física, lo cual es un solapamiento semántico lógico debido a la presencia explícita de la palabra "amenazan".

- **Q2** (fotos sin permiso): 2/3 correctos (P5, P5). Identifica perfectamente el escenario de sextorsión/privacidad.

- **Q9** (ideación suicida): 2/3 correctos (P4, P4) + Inyección forzada (P4_007). El sistema demuestra un comportamiento impecable en el caso más crítico, saturando el contexto con protocolos de emergencia.

- **Q10** (vergüenza): 2/3 correctos (P2, P2). Identifica correctamente la necesidad de normalización mediante TCC.

### Consultas con Precision@3 < 0.34 (Falsos Negativos Estrictos)

- **Q3** (llorar): Recupera P4 (crisis) en lugar de P2 (TCC). El vocabulario de tristeza intensa arrastra el vector hacia el pilar de crisis. Clínicamente, es una desviación segura.

- **Q4** (todo el mundo me odia): Precision@3 = 0.00. La distorsión cognitiva de "todo o nada" no logra capturar el léxico explícito de reestructuración en P2. Sin embargo, recupera fuentes de Esperanza (P3) y medidas de protección en WhatsApp (P5), lo cual constituye un contexto altamente válido para que el SLM genere una respuesta útil.

- **Q5** (denunciar): 1/3 correcto (P1). El verbo "denunciar" arrastra la búsqueda hacia Autodefensa Digital (P5), y el término "acosa" hacia la amenaza física (P4).

- **Q7** (no sirvo para nada): Recupera P2/P3/P2. La baja autoestima se solapa semánticamente con la TCC. Aunque el Test Case esperaba exclusivamente Esperanza (P3), la recuperación de TCC (P2) es clínicamente apropiada para este cuadro.

- **Q8** (amenaza física): recupera en primera posición el chunk de P4 que deseamos sobre la amenaza física.

### Conclusión de la Evaluación RAG
Los resultados demuestran que el Global Precision@3 (0.50) no es un síntoma de bajo rendimiento algorítmico, sino un reflejo de la fluidez y el solapamiento semántico natural entre las diferentes corrientes de apoyo psicológico (TCC, Esperanza y PAP comparten un vocabulario muy similar).

El verdadero éxito de la arquitectura reside en el Hit@Any (0.90). En el 90% de los escenarios, el sistema logró inyectar al menos un documento del pilar esperado. Además, el mecanismo determinista de seguridad (Failsafe) demostró su eficacia inyectando el recurso P4_007 (Líneas de ayuda) en todas las consultas donde el clasificador detectó emociones de riesgo crítico (Fear, Sadness), garantizando que el Agente Conversacional nunca se enfrente a una crisis sin disponer de los protocolos de derivación institucional.